# ESG Data Pipeline
The full pipeline: fetch → chunk → save to `/data`.

In [13]:
import requests, time, re, os, json
from bs4 import BeautifulSoup

HEADERS    = {"User-Agent": "hauglv@usi.ch"}
OUTPUT_DIR = "data"

COMPANY_CIKS = {
    # Bank
    "JPMorgan":          "0000019617",
    "Goldman":           "0000886982",
    "BankOfAmerica":     "0000070858",
    "WellsFargo":        "0000072971",
    "Citigroup":         "0000831001",
    "MorganStanley":     "0000895421",
    "USBancorp":         "0000036104",
    "PNCFinancial":      "0000713676",
    # Capital management
    "BlackRock":         "0001364742",
    "StateStreet":       "0000093751",
    "CharlesSchwab":     "0000316709",
    "BerkshireHathaway": "0001067983",
    "TRowePrice":        "0001113169",
    # Insurance
    "MetLife":           "0001099219",
    "PrudentialFinancial":"0001137774",
    "Aflac":             "0000004977",
    "Allstate":          "0000040729",
    "Travelers":         "0000086312",
    # Fintech
    "Visa":              "0001403161",
    "Mastercard":        "0001141788",
    "PayPal":            "0001633917",
    "Fiserv":            "0000798354",
    # Credit
    "AmericanExpress":   "0000004962",
    "CapitalOne":        "0000927628",
    "DiscoverFinancial": "0001393612",
    # Stock exchange/infrastructure
    "ICE":               "0001571949",
    "Nasdaq":            "0001120193",
    "CMEGroup":          "0001156375",
}

ESG_KEYWORDS = [
    "carbon", "emission", "climate", "greenhouse", "renewable", "energy",
    "waste", "water", "pollution", "biodiversity", "sustainability", "net zero",
    "fossil", "solar", "wind", "environmental",
    "employee", "diversity", "inclusion", "gender", "health", "safety",
    "community", "human rights", "labour", "labor", "training", "wellbeing",
    "workforce", "social",
    "board", "governance", "executive", "compensation", "audit", "compliance",
    "ethics", "transparency", "shareholder", "director", "risk management",
    "anti-corruption", "whistleblower",
]
print(f"Loaded {len(COMPANY_CIKS)} companies.")


Loaded 28 companies.


In [14]:
# clean_url: removes EDGAR's JavaScript display prefix (/ix?doc=) so we get the HTML file itself.
def clean_url(url):
    if "/ix?doc=" in url:
        url = "https://www.sec.gov" + url.split("/ix?doc=")[-1]
    return url

# et_10k_filings: queries the EDGAR API for all filings for a company and returns only 10-K reports with date and stock number.
def get_10k_filings(cik):
    r = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=HEADERS)
    r.raise_for_status()
    d = r.json()["filings"]["recent"]
    return [{"filingDate": date, "accessionNumber": acc}
            for f, date, acc in zip(d["form"], d["filingDate"], d["accessionNumber"])
            if f == "10-K"]

# get_readable_doc_url: opens the submission index, scores all the files and returns the URL of the annual report itself (not XBRL attachments, definitions, etc.).
def get_readable_doc_url(cik, accession):
    cik_int   = int(cik)
    acc_clean = accession.replace("-", "")
    index_url = (f"https://www.sec.gov/Archives/edgar/data/{cik_int}/"
                 f"{acc_clean}/{accession}-index.htm")
    r = requests.get(index_url, headers=HEADERS)
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.content, "html.parser")
    candidates = []
    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) < 3: continue
        link_tag = cells[2].find("a")
        doc_type = cells[3].get_text(strip=True) if len(cells) > 3 else ""
        desc     = cells[1].get_text(strip=True).lower() if len(cells) > 1 else ""
        if not link_tag: continue
        href     = link_tag.get("href", "")
        filename = href.split("/")[-1].lower().split("?")[0]
        skip = [".xml", ".xsd", ".json", "r2.htm", "r3.htm",
                "r4.htm", "r5.htm", "_cal", "_def", "_lab", "_pre"]
        if any(p in filename for p in skip): continue
        if not (filename.endswith(".htm") or filename.endswith(".html")): continue
        score = 0
        if doc_type.upper() in ("10-K", "10-K/A"): score += 20
        if "annual report" in desc: score += 10
        if "10-k" in desc: score += 5
        score -= len(filename)
        full_url = clean_url("https://www.sec.gov" + href if href.startswith("/") else href)
        candidates.append((score, full_url, filename))
    if not candidates: return None
    candidates.sort(reverse=True)
    return candidates[0][1]


# fetch_10k_text:
#  puts it all together: fetches report, downloads HTML, removes script/style tags and returns plain text. Waits 0.6s between calls so as not to overload EDGAR.
def fetch_10k_text(cik, max_filings=1):
    results = []
    for filing in get_10k_filings(cik)[:max_filings]:
        print(f"  Filing: {filing['filingDate']}")
        doc_url = get_readable_doc_url(cik, filing["accessionNumber"])
        if not doc_url: continue
        time.sleep(0.6)
        r = requests.get(doc_url, headers=HEADERS)
        if r.status_code != 200: continue
        if "enable javascript" in r.text.lower()[:300]:
            print("  Warning: got JS page, skipping.")
            continue
        soup = BeautifulSoup(r.content, "html.parser")
        for tag in soup(["script", "style", "head", "nav"]): tag.decompose()
        text = re.sub(r"\s{3,}", "  ", soup.get_text(separator=" ", strip=True))
        print(f"  Extracted {len(text):,} characters.")
        results.append({"filingDate": filing["filingDate"], "text": text})
        time.sleep(0.6)
    return results

print("EDGAR functions ready.")


EDGAR functions ready.


In [15]:
def is_xbrl_noise(chunk: str) -> bool:
    """
    Returns True if the chunk looks like Inline XBRL metadata.
    Filters on both ratio (>15%) AND absolute count (>8 XBRL tokens).
    """
    words = chunk.split()
    if not words:
        return True
    xbrl_count = sum(
        1 for w in words
        if "us-gaap:" in w
        or "fasb.org" in w
        or "xbrli:" in w
        or "bac:" in w
        or "srt:" in w
        or "iso4217:" in w
        or "utr:" in w
        or "stpr:" in w
        or "country:" in w
        or (len(w) > 25 and "Member" in w)
        or (len(w) > 20 and w.count(":") >= 2)
        or (w.startswith("0000") and len(w) in (7, 10))  # CIK numbers
    )
    ratio = xbrl_count / len(words)
    # Noise if ratio high OR too many absolute XBRL tokens
    return ratio > 0.15 or xbrl_count > 8

def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    all_chunks = [" ".join(words[i:i+chunk_size])
                  for i in range(0, len(words), chunk_size - overlap)]
    clean    = [c for c in all_chunks if not is_xbrl_noise(c)]
    relevant = [c for c in clean if any(kw in c.lower() for kw in ESG_KEYWORDS)]
    print(f"  {len(all_chunks)} chunks → {len(clean)} after XBRL filter → {len(relevant)} ESG relevant")
    return relevant

# saves the filtered chuncks in a JSON-file in \data
def save_chunks(company, filing_date, chunks):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, f"{company}_{filing_date}_chunks.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"company": company, "filingDate": filing_date,
                   "chunkCount": len(chunks), "chunks": chunks},
                  f, indent=2, ensure_ascii=False)
    print(f"  Saved → {path}")

print("Chunker functions ready.")


Chunker functions ready.


In [17]:
# checking if the data folder already has files
existing = set()
if os.path.exists(OUTPUT_DIR):
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith("_chunks.json"):
            company = f.split("_")[0]
            existing.add(company)

if existing:
    print(f"NOTE: These companies already have data and are skipped: {sorted(existing)}")
    print("Delete the data/ folder manually if we want to retrieve it again..\n")

COMPANIES = [c for c in COMPANY_CIKS.keys() if c not in existing]
print(f"Running pipeline for {len(COMPANIES)} companies...\n")


for company in COMPANIES:
    print(f"\n{sep}\n{company}\n{sep}")
    cik = COMPANY_CIKS.get(company)
    if not cik:
        print("CIK not found, skipping.")
        continue
    filings = fetch_10k_text(cik, max_filings=1)
    for filing in filings:
        chunks = chunk_text(filing["text"])
        save_chunks(company, filing["filingDate"], chunks)

print("\nPipeline complete! Check /data-file.")


Running pipeline for 28 companies...



NameError: name 'sep' is not defined

In [ ]:
# Preview the first saved file
files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".json")]
if not files:
    print("No files yet, run the pipeline first.")
else:
    with open(os.path.join(OUTPUT_DIR, files[0])) as f:
        d = json.load(f)
    print(f"Company    : {d['company']}")
    print(f"Filing date: {d['filingDate']}")
    print(f"Chunks     : {d['chunkCount']}")
    print(f"\nFirst chunk:")
    print(d['chunks'][0][:500] if d['chunks'] else "No chunks.")


Company    : MetLife
Filing date: 2026-02-19
Chunks     : 218

First chunk:
filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months (or for such shorter period that the registrant was required to file such reports), and (2) has been subject to such filing requirements for the past 90 days. Yes þ No ¨ Indicate by check mark whether the registrant has submitted electronically every Interactive Data File required to be submitted pursuant to Rule 405 of Regulation S-T (§ 232.405 of this chapter) during the preceding 12 months (or f
